In [1]:
from dotenv import load_dotenv
load_dotenv()

from decouple import config
api_key=config("OPEN_ROUTER_API_KEY")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="openrouter/owl-alpha",
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

In [3]:
from typing import List, Dict, Union, Literal
from pydantic import BaseModel, Field


class UserInput(BaseModel):
    total_hours: int = Field(description="Total working hours entered by the user")
    max_shift_hours: int = Field(description="Maximum hours worked in a single shift entered by the user")
    number_of_weeks: int = Field(description="Number of weeks entered by the user")
    agreement_type: str = Field(description="Agreement type selected by the user")


class RuleCondition(BaseModel):
    field: str = Field(description="User input field or calculated metric being evaluated")
    operator: str = Field(description="Comparison operator extracted from the rule")
    value: Union[int, float, str] = Field(description="Threshold value defined in the rule")


class Citation(BaseModel):
    source_document: str = Field(description="Name of the uploaded PDF")
    title: str = Field(description="Chapter or title where the rule is found")
    section: str = Field(description="Section containing the rule")


class Rule(BaseModel):
    rule_id: str = Field(description="Generated unique identifier for the rule")
    rule_name: str = Field(description="Rule name extracted from the PDF")
    condition: RuleCondition
    citation: Citation



class Violation(BaseModel):
    rule_id: str
    rule_name: str
    legal_status: Literal["LEGAL", "NOT_LEGAL"]
    reason: str
    actual_value: Union[int, float, str]
    expected_value: Union[int, float, str]
    citation: Citation


class ComplianceResult(BaseModel):
    overall_status: Literal["LEGAL", "NOT_LEGAL"]
    evaluated_values: Dict[str, Union[int, float, str]] = Field(description="Calculated values used during compliance evaluation" )
    violations: List[Violation]

In [ ]:
from pypdf import PdfReader

pdf_path = "govt_law.pdf"
reader = PdfReader(pdf_path)

document_text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        document_text += page_text + "\n"

In [ ]:
RULE_EXTRACTION_PROMPT = f"""
You are a compliance rule extraction assistant.

Your task is to read the provided legal/company policy text and extract only working-hour related compliance rules.

Extract rules related to:
- normal weekly working hours
- average weekly working hours
- maximum shift hours
- overtime limits
- breaks
- night work
- Sunday/holiday work
- rest periods

Return the output strictly as a list of Rule objects matching this Pydantic schema.



Important rules:
1. Do not invent any rule.
2. Extract only rules clearly present in the text.
3. Use exact legal threshold values from the text.
4. If no valid working-hour rule is found, return an empty list.
5. Keep citation aligned with source_document, title, and section only.
6. Do not add explanations outside the structured output.

Additional extraction requirement:

The condition.field must match only one of these calculator output fields:
- average_weekly_hours
- max_shift_hours
- total_hours
- number_of_weeks

Mapping rules:
- Any weekly working-hour limit must use field = "average_weekly_hours".
- Any daily working-hour, 24-hour, or shift limit must use field = "max_shift_hours".
- Any total entered hours limit must use field = "total_hours".
- Number of weeks related rules must use field = "number_of_weeks".

The condition.operator must use only:
<, <=, >, >=, ==, !=

The condition.value must be a clean numeric value only.

Do not return long text inside condition.value.
Do not create new field names like normal_working_hours or working_hours.

Examples:
- "40 hours per seven days" → field="average_weekly_hours", operator="<=", value=40
- "48 hours per week" → field="average_weekly_hours", operator="<=", value=48
- "9 hours per 24 hours" → field="max_shift_hours", operator="<=", value=9

Text:
{document_text}
"""

class RuleList(BaseModel):
    rules: List[Rule]

structured_llm = llm.with_structured_output(RuleList)
response = structured_llm.invoke(RULE_EXTRACTION_PROMPT)
rules = response.rules

In [6]:
for rule in rules:
    print(rule.model_dump_json(indent=4))

{
    "rule_id": "10-4-1",
    "rule_name": "Normal working hours limit",
    "condition": {
        "field": "max_shift_hours",
        "operator": "<=",
        "value": 9
    },
    "citation": {
        "source_document": "Working Environment Act",
        "title": "Act of 17 June 2005 No. 62 relating to working environment, working hours and employment protection, etc.",
        "section": "Section 10-4(1)"
    }
}
{
    "rule_id": "10-4-1b",
    "rule_name": "Normal weekly working hours limit",
    "condition": {
        "field": "average_weekly_hours",
        "operator": "<=",
        "value": 40
    },
    "citation": {
        "source_document": "Working Environment Act",
        "title": "Act of 17 June 2005 No. 62 relating to working environment, working hours and employment protection, etc.",
        "section": "Section 10-4(1)"
    }
}
{
    "rule_id": "10-4-2",
    "rule_name": "Passive work extended weekly hours",
    "condition": {
        "field": "average_weekly_hour

In [7]:
def calculator(user_input: UserInput) -> dict:
    average_weekly_hours = round(
        user_input.total_hours / user_input.number_of_weeks,
        2
    )

    return {
        "total_hours": user_input.total_hours,
        "max_shift_hours": user_input.max_shift_hours,
        "number_of_weeks": user_input.number_of_weeks,
        "agreement_type": user_input.agreement_type,
        "average_weekly_hours": average_weekly_hours
    }

In [8]:
def compare_values(actual_value, operator, expected_value):

    if operator == "<=":
        return actual_value <= expected_value

    elif operator == "<":
        return actual_value < expected_value

    elif operator == ">=":
        return actual_value >= expected_value

    elif operator == ">":
        return actual_value > expected_value

    elif operator in ["=", "=="]:
        return actual_value == expected_value

    elif operator == "!=":
        return actual_value != expected_value

    return True

In [9]:
def check_compliance(
    user_input: UserInput,
    rules: list[Rule]
) -> ComplianceResult:

    evaluated_values = calculator(user_input)

    violations = []

    for rule in rules:
        
        field = rule.condition.field
        operator = rule.condition.operator
        expected_value = rule.condition.value

        if field not in evaluated_values:
            continue

        actual_value = evaluated_values[field]

        is_legal = compare_values(
            actual_value,
            operator,
            expected_value
        )

        if not is_legal:

            violations.append(
                Violation(
                    rule_id=rule.rule_id,
                    rule_name=rule.rule_name,
                    legal_status="NOT_LEGAL",
                    reason=(
                        f"{field} = {actual_value} "
                        f"violates rule {operator} {expected_value}"
                    ),
                    actual_value=actual_value,
                    expected_value=expected_value,
                    citation=rule.citation
                )
            )

    overall_status = (
        "LEGAL"
        if len(violations) == 0
        else "NOT_LEGAL"
    )

    return ComplianceResult(
        overall_status=overall_status,
        evaluated_values=evaluated_values,
        violations=violations
    )

In [10]:
user_input = UserInput(
    total_hours=400,
    max_shift_hours=12,
    number_of_weeks=3,
    agreement_type="government_law"
)

result = check_compliance(user_input, rules)

print(result.model_dump_json(indent=4))

{
    "overall_status": "NOT_LEGAL",
    "evaluated_values": {
        "total_hours": 400,
        "max_shift_hours": 12,
        "number_of_weeks": 3,
        "agreement_type": "government_law",
        "average_weekly_hours": 133.33
    },
    "violations": [
        {
            "rule_id": "10-4-1",
            "rule_name": "Normal working hours limit",
            "legal_status": "NOT_LEGAL",
            "reason": "max_shift_hours = 12 violates rule <= 9",
            "actual_value": 12,
            "expected_value": 9,
            "citation": {
                "source_document": "Working Environment Act",
                "title": "Act of 17 June 2005 No. 62 relating to working environment, working hours and employment protection, etc.",
                "section": "Section 10-4(1)"
            }
        },
        {
            "rule_id": "10-4-1b",
            "rule_name": "Normal weekly working hours limit",
            "legal_status": "NOT_LEGAL",
            "reason": "average